# 血液抹片細胞自動辨識與計數系統
> 影像處理 114-2 ｜ 期末專案

**使用技術**：OpenCV · Adaptive Thresholding · Morphological Operations · Watershed · Streamlit

| 區塊 | 功能 |
|------|------|
| §0 Setup | 安裝依賴、載入資料集 |
| §1 Utils | 輔助函式 |
| §2 Preprocess | 前處理 Pipeline |
| §3 Detect | 輪廓偵測 |
| §4 Classify | RBC / WBC / Platelet 分類 |
| §5 Watershed | 重疊細胞切分 |
| §6 Pipeline | 總控函式 |
| §7 Evaluation | 5 張圖固定參數評估 |
| §8 App | Streamlit 互動式 App |

## §0 環境設定

In [ ]:
# §0 安裝依賴套件
!pip install -q streamlit

# 下載 cloudflared（免費 tunnel，不需要帳號）
import os
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 下載 BCCD Dataset（若已存在則跳過）
if not os.path.exists('/content/BCCD_Dataset'):
    !git clone https://github.com/Shenggan/BCCD_Dataset.git /content/BCCD_Dataset
else:
    print('BCCD Dataset 已存在')

# 載入常用套件
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xml.etree.ElementTree as ET
import glob
from dataclasses import dataclass

print('✅ 環境設定完成')

## §1 輔助函式

In [ ]:
# §1 輔助函式

def show(title, img):
    '''顯示影像於 notebook（支援灰階與彩色）'''
    plt.figure(figsize=(6, 4))
    plt.title(title)
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


def show_stages(stages_dict, stage_order, stage_labels):
    '''並排顯示多個前處理階段'''
    n = len(stage_order)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for ax, key in zip(axes, stage_order):
        img = stages_dict[key]
        if len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(stage_labels[key])
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def overlay_contours(img, contours_dict):
    '''
    對每個偵測到的細胞畫最小外接圓（而非原始輪廓），視覺上更清晰。
    RBC：紅色，WBC：綠色，Platelet：藍色
    '''
    result = img.copy()
    colors = {'rbc': (0, 0, 255), 'wbc': (0, 255, 0), 'platelet': (255, 0, 0)}
    for cell_type, contours in contours_dict.items():
        for cnt in contours:
            (x, y), r = cv2.minEnclosingCircle(cnt)
            cv2.circle(result, (int(x), int(y)), int(r), colors[cell_type], 2)
    return result


def encode_image(img):
    '''將 BGR 影像編碼為 PNG bytes，供下載使用'''
    _, buffer = cv2.imencode('.png', img)
    return buffer.tobytes()


print('✅ Utils 載入完成')

## §2 影像前處理 Pipeline

**步驟**：灰階 → Gaussian Blur → Adaptive Thresholding → Morphological Closing

- **Adaptive Thresholding**：血液抹片光照不均，全域閾值效果差；Adaptive 以局部區塊均值為基準，對不均勻背景更穩健
- **Morphological Closing**：先膨脹再侵蝕，填補細胞內部因染色不均產生的空洞

In [ ]:
# §2 前處理 Pipeline

@dataclass
class PreprocessParams:
    gaussian_ksize: int = 5    # Gaussian kernel 大小（必須為奇數）
    adaptive_block: int = 71   # Adaptive threshold 區塊大小；71 ≈ 1.7× 細胞直徑，
                                # 使局部均值含足夠背景像素，讓 RBC 淡色中央也落於均值以下
    adaptive_c:     int = 2    # 從區塊均值減去的常數
    morph_ksize:    int = 5    # 形態學 kernel 大小
    morph_iter:     int = 1    # 形態學操作迭代次數


def preprocess(img, params):
    '''
    前處理 pipeline：灰階 → 高斯模糊 → Adaptive Thresholding → Morphological Closing → 輪廓填充
    回傳各階段影像 dict，可用於 App 可視化
    '''
    # 1. 灰階轉換
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. 高斯模糊：抑制高頻雜訊
    blurred = cv2.GaussianBlur(
        gray, (params.gaussian_ksize, params.gaussian_ksize), 0
    )

    # 3. Adaptive Thresholding：以局部均值為基準處理不均勻光照
    #    block=71 確保局部窗口含足夠背景，使 RBC 淡色中央（pale center）
    #    也落於局部均值以下，偵測到更完整的細胞實心區域（非單純環形膜）
    thresh = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        params.adaptive_block,
        params.adaptive_c
    )

    # 4. Morphological Closing：橢圓形 kernel 填補細胞輪廓上的小缺口
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (params.morph_ksize, params.morph_ksize)
    )
    morphed = cv2.morphologyEx(
        thresh, cv2.MORPH_CLOSE, kernel, iterations=params.morph_iter
    )

    # 5. 輪廓填充：將閉合的環形輪廓填充為實心圓盤
    #    （Flood Fill 在 morph_iter≥2 後角落像素可能變白，導致填充失效，
    #      改用 RETR_EXTERNAL 輪廓的 FILLED 繪製更穩定）
    cnts, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(morphed)
    cv2.drawContours(filled, cnts, -1, 255, cv2.FILLED)
    morphed = filled

    return {
        'original': img,
        'gray':     gray,
        'blurred':  blurred,
        'thresh':   thresh,
        'morphed':  morphed,
    }


print('✅ Preprocess 載入完成')

In [ ]:
# §2 Demo：顯示前處理各階段結果
sample_img = cv2.imread('/content/BCCD_Dataset/BCCD/JPEGImages/BloodImage_00001.jpg')

if sample_img is None:
    print('⚠️  請先執行 §0 Setup 以載入資料集')
else:
    stages = preprocess(sample_img, PreprocessParams())
    order = ['original', 'gray', 'blurred', 'thresh', 'morphed']
    labels = {
        'original': '原圖',
        'gray':     '灰階',
        'blurred':  'Gaussian 模糊',
        'thresh':   'Adaptive Thresholding',
        'morphed':  'Morphological Closing',
    }
    show_stages(stages, order, labels)

## §3 細胞偵測

從 Morphological Closing 結果找輪廓，計算每個輪廓的：
- **面積** (`area`)：像素數量
- **圓形度** (`circularity`)：`4π·A / P²`，完美圓形 = 1，不規則形狀趨近 0

In [ ]:
# §3 細胞偵測：輪廓分析

def detect_cells(binary, min_area=50):
    '''
    從二值影像找出細胞輪廓，計算幾何特徵
    min_area：過濾背景雜訊用的最小面積閾值（px²）
    '''
    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cells = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter < 1:
            continue
        # 圓形度：4πA/P²，裁切到 [0, 1] 避免數值誤差
        circularity = min(4 * np.pi * area / (perimeter ** 2), 1.0)
        cells.append({
            'contour':     cnt,
            'area':        area,
            'perimeter':   perimeter,
            'circularity': circularity,
            'bbox':        cv2.boundingRect(cnt),
        })
    return cells


print('✅ Detect 載入完成')

## §4 細胞分類

**分類規則（面積 + 圓形度）**

| 細胞 | 面積（px²） | 圓形度 | 顏色 |
|------|------------|--------|------|
| Platelet（血小板） | < 300 | ≥ 0.5 | 藍 |
| WBC（白血球） | > 2000 | ≥ 0.6 | 綠 |
| RBC（紅血球） | 其餘 | 任意 | 紅 |

In [ ]:
# §4 細胞分類：面積 + HSV 飽和度規則

# 分類閾值
PLATELET_AREA_MAX  = 300    # 血小板面積上限（px²）
PLATELET_CIRC_MIN  = 0.5    # 血小板最小圓形度
WBC_AREA_MIN       = 2000   # 白血球面積下限（px²）
WBC_SAT_MIN        = 70     # WBC 的 HSV 飽和度下限
                             # WBC 以 Giemsa 染色呈紫色（S ≈ 80–95）
                             # RBC 淡粉色（S ≈ 17–44），間距明顯可區分


def classify(cells, img):
    '''
    以面積、圓形度、HSV 飽和度分類 RBC / WBC / Platelet。

    WBC 識別改用顏色特徵：Giemsa 染色使 WBC 呈深紫色（高飽和度），
    純以面積/圓形度判斷時，大型 RBC 細胞叢會誤判為 WBC；
    加入飽和度門檻後誤判率大幅降低。
    '''
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    result = {'rbc': [], 'wbc': [], 'platelet': []}

    for cell in cells:
        area = cell['area']
        circ = cell['circularity']
        cnt  = cell['contour']

        # 計算此區域的 HSV 平均飽和度
        mask = np.zeros(img.shape[:2], np.uint8)
        cv2.drawContours(mask, [cnt], -1, 255, cv2.FILLED)
        mean_sat = cv2.mean(hsv[:, :, 1], mask=mask)[0]

        if area < PLATELET_AREA_MAX and circ >= PLATELET_CIRC_MIN:
            result['platelet'].append(cnt)
        elif area >= WBC_AREA_MIN and mean_sat >= WBC_SAT_MIN:
            # 大面積 + 高飽和度 = WBC（紫色染色）
            result['wbc'].append(cnt)
        else:
            # RBC 數量最多，作為預設類別
            result['rbc'].append(cnt)
    return result


print('✅ Classify 載入完成')

## §5 Watershed 重疊細胞切分

**為什麼選 Watershed？**  
血液抹片中 RBC 常相鄰排列，形態學無法分離。Watershed 以 Distance Transform 的峰值為「水源」向外擴散，在相鄰細胞之間自動找到分水嶺邊界。

**步驟**
1. 膨脹取得確定背景
2. Distance Transform → 峰值為確定前景（細胞核心）
3. 未確定區域 = 背景 - 前景（相鄰邊界）
4. Watershed 標記邊界（-1）

In [ ]:
# §5 Watershed 重疊細胞切分

def watershed_split(original, binary):
    '''
    使用 Distance Transform + Watershed 分離相鄰或重疊細胞
    回傳：(markers, dist_transform)
    '''
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    # 確定背景：膨脹後的二值遮罩
    sure_bg = cv2.dilate(binary, kernel, iterations=3)

    # Distance Transform：每個前景像素到最近背景的距離
    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)

    # 確定前景：距離峰值 50% 以上的區域為細胞核心
    if dist.max() == 0:
        return np.ones_like(binary, dtype=np.int32), dist
    _, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)

    # 未確定區域（相鄰邊界）
    unknown = cv2.subtract(sure_bg, sure_fg)

    # 連通域標記（每個細胞核心一個唯一 ID）
    _, markers = cv2.connectedComponents(sure_fg)
    markers += 1              # 背景從 1 開始，保留 0 給未確定區域
    markers[unknown == 255] = 0

    # Watershed：邊界處標為 -1
    markers = cv2.watershed(original, markers)

    return markers, dist


print('✅ Watershed 載入完成')

## §6 總控 Pipeline

In [ ]:
# §6 總控 Pipeline：整合前處理 → Watershed → 偵測 → 分類 → 視覺化

def run_pipeline(img, params):
    '''
    完整處理流程，回傳包含各階段結果的 dict
    執行順序：preprocess → watershed_split → detect_cells → classify → overlay
    '''
    # 前處理
    stages = preprocess(img, params)

    # Watershed 切分重疊細胞
    labels, dist = watershed_split(img, stages['morphed'])

    # 從 Watershed 結果重建二值遮罩（各切分區域）
    ws_binary = np.zeros_like(stages['morphed'])
    ws_binary[labels > 1] = 255

    # 輪廓偵測（min_area=30，寬鬆一點以偵測較小細胞）
    cells = detect_cells(ws_binary, min_area=30)

    # 分類（傳入原圖以計算 HSV 飽和度識別 WBC）
    classified = classify(cells, img)

    # 繪製標註圖（最小外接圓，比原始輪廓視覺上更清晰）
    annotated = overlay_contours(img.copy(), classified)

    return {
        'stages':     stages,
        'cells':      cells,
        'counts':     {
            'rbc':      len(classified['rbc']),
            'wbc':      len(classified['wbc']),
            'platelet': len(classified['platelet']),
        },
        'classified': classified,
        'overlay':    annotated,
        'labels':     labels,
        'dist':       dist,
    }


# 快速測試（使用第一張範例圖）
sample_img = cv2.imread('/content/BCCD_Dataset/BCCD/JPEGImages/BloodImage_00001.jpg')
if sample_img is not None:
    result = run_pipeline(sample_img, PreprocessParams())
    print('偵測結果：', result['counts'])
    show('標註結果（紅:RBC／綠:WBC／藍:Platelet）', result['overlay'])
else:
    print('⚠️  請先執行 §0 Setup')

## §7 固定參數評估（5 張圖通用性驗證）

驗證「5 張圖通用」：固定 `PreprocessParams()` 預設值，不手動調整，對 5 張不同圖片評估誤差。

In [ ]:
# §7 評估函式

def parse_gt(xml_path):
    '''解析 BCCD Dataset XML annotation，取得各類細胞真實數量'''
    tree = ET.parse(xml_path)
    root = tree.getroot()
    counts = {'rbc': 0, 'wbc': 0, 'platelet': 0}
    label_map = {'RBC': 'rbc', 'WBC': 'wbc', 'Platelets': 'platelet'}
    for obj in root.findall('object'):
        name = obj.find('name').text
        key = label_map.get(name)
        if key:
            counts[key] += 1
    return counts


def eval_table(pipeline_fn, params, test_images):
    '''
    對多張測試圖執行固定參數 pipeline，輸出預測 vs Ground Truth 誤差表
    test_images：list of (img_path, xml_path)
    '''
    rows = []
    for img_path, xml_path in test_images:
        img = cv2.imread(img_path)
        if img is None:
            print(f'⚠️  無法讀取: {img_path}')
            continue
        result = pipeline_fn(img, params)
        pred = result['counts']
        gt   = parse_gt(xml_path)

        def err(p, g):
            return round(abs(p - g) / g * 100, 1) if g > 0 else 0.0

        rows.append({
            'image':       os.path.basename(img_path),
            'rbc_pred':    pred['rbc'],      'rbc_gt':  gt['rbc'],      'rbc_err%':  err(pred['rbc'],      gt['rbc']),
            'wbc_pred':    pred['wbc'],      'wbc_gt':  gt['wbc'],      'wbc_err%':  err(pred['wbc'],      gt['wbc']),
            'plt_pred':    pred['platelet'], 'plt_gt':  gt['platelet'], 'plt_err%':  err(pred['platelet'], gt['platelet']),
        })
    return pd.DataFrame(rows)


print('✅ Evaluation 載入完成')

In [ ]:
# §7 執行 5 張圖評估
BCCD_IMG = '/content/BCCD_Dataset/BCCD/JPEGImages'
BCCD_ANN = '/content/BCCD_Dataset/BCCD/Annotations'

test_names = [
    'BloodImage_00001', 'BloodImage_00002', 'BloodImage_00003',
    'BloodImage_00004', 'BloodImage_00005',
]
test_images = [
    (f'{BCCD_IMG}/{n}.jpg', f'{BCCD_ANN}/{n}.xml') for n in test_names
]

# 固定預設參數，不手動調整
fixed_params = PreprocessParams()

df = eval_table(run_pipeline, fixed_params, test_images)
print('=== 固定參數評估結果 ===')
print(df.to_string(index=False))

# 印出各類平均誤差
print()
print(f'RBC 平均誤差：    {df["rbc_err%"].mean():.1f}%')
print(f'WBC 平均誤差：    {df["wbc_err%"].mean():.1f}%')
print(f'Platelet 平均誤差：{df["plt_err%"].mean():.1f}%')

## §8 Streamlit 互動式 App

**流程**：
1. 執行 §8a：將 `app.py` 寫入 Colab 環境
2. 執行 §8b：用 cloudflared 建立公開 URL（不需要帳號）

In [ ]:
%%writefile app.py
# app.py — 血液抹片細胞自動辨識 Streamlit App
import cv2
import numpy as np
import streamlit as st
import os
import glob
from dataclasses import dataclass


# ── 參數結構 ────────────────────────────────────────────────────────────────
@dataclass
class PreprocessParams:
    gaussian_ksize: int = 5
    adaptive_block: int = 71   # 71 ≈ 1.7× 細胞直徑，讓 RBC 淡色中央也被偵測
    adaptive_c:     int = 2
    morph_ksize:    int = 5
    morph_iter:     int = 1


# ── 核心演算法（與 notebook §2–§6 相同）─────────────────────────────────────

def preprocess(img, params):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(
        gray, (params.gaussian_ksize, params.gaussian_ksize), 0
    )
    thresh = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        params.adaptive_block,
        params.adaptive_c
    )
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (params.morph_ksize, params.morph_ksize)
    )
    morphed = cv2.morphologyEx(
        thresh, cv2.MORPH_CLOSE, kernel, iterations=params.morph_iter
    )
    # 輪廓填充：比 Flood Fill 更穩定（不受角落像素影響）
    cnts, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(morphed)
    cv2.drawContours(filled, cnts, -1, 255, cv2.FILLED)
    morphed = filled
    return {
        'original': img, 'gray': gray, 'blurred': blurred,
        'thresh': thresh, 'morphed': morphed,
    }


def detect_cells(binary, min_area=30):
    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cells = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter < 1:
            continue
        circularity = min(4 * np.pi * area / (perimeter ** 2), 1.0)
        cells.append({
            'contour': cnt, 'area': area,
            'perimeter': perimeter, 'circularity': circularity,
            'bbox': cv2.boundingRect(cnt),
        })
    return cells


def classify(cells, img):
    '''WBC 以 HSV 飽和度識別（Giemsa 紫色染色 S≈80–95，RBC 淡粉色 S≈17–44）'''
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    result = {'rbc': [], 'wbc': [], 'platelet': []}
    for cell in cells:
        area, circ, cnt = cell['area'], cell['circularity'], cell['contour']
        mask = np.zeros(img.shape[:2], np.uint8)
        cv2.drawContours(mask, [cnt], -1, 255, cv2.FILLED)
        mean_sat = cv2.mean(hsv[:, :, 1], mask=mask)[0]
        if area < 300 and circ >= 0.5:
            result['platelet'].append(cnt)
        elif area >= 2000 and mean_sat >= 70:
            result['wbc'].append(cnt)
        else:
            result['rbc'].append(cnt)
    return result


def watershed_split(original, binary):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    sure_bg = cv2.dilate(binary, kernel, iterations=3)
    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    if dist.max() == 0:
        return np.ones_like(binary, dtype=np.int32), dist
    _, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    unknown = cv2.subtract(sure_bg, sure_fg)
    _, markers = cv2.connectedComponents(sure_fg)
    markers += 1
    markers[unknown == 255] = 0
    markers = cv2.watershed(original, markers)
    return markers, dist


def overlay_contours(img, contours_dict):
    '''畫最小外接圓，視覺上比原始輪廓更清晰'''
    result = img.copy()
    colors = {'rbc': (0, 0, 255), 'wbc': (0, 255, 0), 'platelet': (255, 0, 0)}
    for cell_type, contours in contours_dict.items():
        for cnt in contours:
            (x, y), r = cv2.minEnclosingCircle(cnt)
            cv2.circle(result, (int(x), int(y)), int(r), colors[cell_type], 2)
    return result


def run_pipeline(img_bytes, gk, ab, ac, mk):
    img = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
    params = PreprocessParams(
        gaussian_ksize=gk, adaptive_block=ab, adaptive_c=ac, morph_ksize=mk
    )
    stages = preprocess(img, params)
    labels, dist = watershed_split(img, stages['morphed'])
    ws_binary = np.zeros_like(stages['morphed'])
    ws_binary[labels > 1] = 255
    cells = detect_cells(ws_binary, min_area=30)
    classified = classify(cells, img)
    annotated = overlay_contours(img.copy(), classified)
    return {
        'img': img, 'stages': stages, 'cells': cells,
        'counts': {
            'rbc': len(classified['rbc']),
            'wbc': len(classified['wbc']),
            'platelet': len(classified['platelet']),
        },
        'classified': classified,
        'overlay': annotated,
        'labels': labels,
        'dist': dist,
    }


def to_rgb(img):
    if len(img.shape) == 2:
        return img
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# ── Streamlit UI ─────────────────────────────────────────────────────────────
st.set_page_config(page_title='血液抹片分析', layout='wide', page_icon='🔬')

st.title('🔬 血液抹片細胞自動辨識與計數系統')
st.caption('影像處理 114-2 ｜ OpenCV · Adaptive Thresholding · Watershed · Streamlit')

DEFAULTS = {'gk': 5, 'ab': 71, 'ac': 2, 'mk': 5}
for k, v in DEFAULTS.items():
    if k not in st.session_state:
        st.session_state[k] = v

# ── Sidebar ──
with st.sidebar:
    st.header('⚙️ 設定')

    src_mode = st.radio('影像來源', ['上傳圖片', '範例圖片'])
    img_bytes = None

    if src_mode == '上傳圖片':
        uploaded = st.file_uploader(
            '選擇血液抹片影像', type=['jpg', 'jpeg', 'png']
        )
        if uploaded:
            img_bytes = uploaded.read()
    else:
        sample_dir = '/content/BCCD_Dataset/BCCD/JPEGImages'
        samples = sorted(
            glob.glob(os.path.join(sample_dir, 'BloodImage_000*.jpg'))
        )[:5]
        if samples:
            idx = st.selectbox(
                '選擇範例圖片', range(len(samples)),
                format_func=lambda i: os.path.basename(samples[i])
            )
            with open(samples[idx], 'rb') as f:
                img_bytes = f.read()
        else:
            st.warning('找不到 BCCD dataset，請先執行 §0 Setup cell')

    st.divider()
    st.subheader('📊 前處理參數')

    if st.button('🔄 Reset 預設值'):
        for k, v in DEFAULTS.items():
            st.session_state[k] = v
        st.rerun()

    gk = st.slider('Gaussian Kernel Size', 3, 11, step=2, key='gk',
                   help='高斯模糊 kernel 大小，越大越平滑')
    ab = st.slider('Adaptive Block Size', 11, 101, step=2, key='ab',
                   help='Adaptive threshold 區塊大小（建議 ≥ 71 以偵測 RBC 實心區域）')
    ac = st.slider('Adaptive C', 1, 15, key='ac',
                   help='從區塊均值減去的常數，越大前景越少')
    mk = st.slider('Morph Kernel Size', 1, 9, step=2, key='mk',
                   help='形態學 kernel 大小')

# ── Main Area ──
if img_bytes is None:
    st.info('👈 請在左側選擇或上傳一張血液抹片影像')
    st.stop()

with st.spinner('處理中...'):
    result = run_pipeline(
        img_bytes,
        st.session_state['gk'], st.session_state['ab'],
        st.session_state['ac'], st.session_state['mk'],
    )

tab1, tab2, tab3 = st.tabs(['🔬 Pipeline 可視化', '🧬 分類結果', '💧 Watershed'])

# Tab 1：前處理各階段視覺化
with tab1:
    st.subheader('影像前處理各階段')
    stages = result['stages']
    stage_items = [
        ('original', '① 原圖'),
        ('gray',     '② 灰階'),
        ('blurred',  '③ Gaussian 模糊'),
        ('thresh',   '④ Adaptive Thresholding'),
        ('morphed',  '⑤ Morph Closing + 輪廓填充'),
    ]
    cols = st.columns(3)
    for i, (key, label) in enumerate(stage_items):
        with cols[i % 3]:
            st.caption(label)
            st.image(to_rgb(stages[key]), use_container_width=True)

    st.divider()
    st.caption('⑥ 輪廓偵測結果（所有偵測到的輪廓以青色標示）')
    contour_vis = result['img'].copy()
    all_cnts = [c['contour'] for c in result['cells']]
    cv2.drawContours(contour_vis, all_cnts, -1, (0, 255, 255), 2)
    st.image(to_rgb(contour_vis), use_container_width=True)

# Tab 2：分類結果
with tab2:
    st.subheader('細胞分類結果')
    col_img, col_stat = st.columns([2, 1])

    with col_img:
        st.image(
            to_rgb(result['overlay']),
            caption='標註圖（🔴 RBC ／ 🟢 WBC ／ 🔵 Platelet）',
            use_container_width=True
        )

    with col_stat:
        counts = result['counts']
        st.metric('🔴 RBC（紅血球）',     counts['rbc'])
        st.metric('🟢 WBC（白血球）',     counts['wbc'])
        st.metric('🔵 Platelet（血小板）', counts['platelet'])
        st.divider()
        st.metric('總計', sum(counts.values()))

    st.divider()
    _, buf = cv2.imencode('.png', result['overlay'])
    st.download_button(
        '⬇️ 下載標註圖 PNG',
        buf.tobytes(), 'annotated.png', 'image/png'
    )

# Tab 3：Watershed 視覺化
with tab3:
    st.subheader('Watershed 重疊細胞切分')
    col1, col2 = st.columns(2)

    with col1:
        st.caption('切分前（Morph Closing + 輪廓填充結果）')
        st.image(result['stages']['morphed'], use_container_width=True)

    with col2:
        ws_vis = result['img'].copy()
        ws_vis[result['labels'] == -1] = [0, 0, 255]
        st.caption('切分後（紅線 = Watershed 邊界）')
        st.image(to_rgb(ws_vis), use_container_width=True)

    st.divider()
    dist_norm = cv2.normalize(
        result['dist'], None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)
    dist_heat = cv2.applyColorMap(dist_norm, cv2.COLORMAP_JET)
    st.caption('Distance Transform 熱圖（暖色 = 細胞核心，冷色 = 邊界）')
    st.image(to_rgb(dist_heat), use_container_width=True)


In [ ]:
# §8b 啟動 Streamlit App（Colab 環境，使用 cloudflared）
import subprocess
import time
import re

# 啟動 Streamlit（背景執行）
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

# 啟動 cloudflared tunnel，從 stderr 讀取公開 URL
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

# 等待並解析 trycloudflare.com URL
url = None
for line in cf_proc.stderr:
    text = line.decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', text)
    if match:
        url = match.group(0)
        break

print(f'✅ Streamlit App 已啟動')
print(f'🌐 公開網址：{url}')
print('點擊連結即可開啟互動式 App（3 分鐘 Demo 用這個 URL）')